# การวิเคราะห์พฤติกรรมการใช้จ่ายของนักศึกษา มข.

## 1. ภาพรวมโครงการ

Notebook สาธารณะนี้เป็นการวิเคราะห์เชิงสำรวจของกลุ่มตัวอย่างนักศึกษา มข. ที่ตอบแบบสอบถามด้วยตนเองโดยสมัครใจ (N=30) รายงานเฉพาะผลสรุปที่อนุมัติแล้ว และไม่ใช่ตัวแทนของนักศึกษา มข. ทั้งหมด

## 2. ชุดข้อมูล

ข้อมูลดิบและข้อมูลประมวลผลระดับผู้ตอบถูกยกเว้นเพื่อความเป็นส่วนตัว Notebook นี้ใช้เฉพาะตาราง aggregate และภาพสรุปที่ปลอดภัยต่อการเผยแพร่ การรัน pipeline แบบครบถ้วนต้องใช้ข้อมูลส่วนตัวภายใต้การควบคุมที่เหมาะสม

In [ ]:
from pathlib import Path
import csv
ROOT = Path.cwd().resolve()
if not (ROOT / 'outputs').exists():
    ROOT = ROOT.parent
TABLES = ROOT / 'outputs' / 'tables'
PUBLIC_FIGURES = ROOT / 'outputs' / 'figures' / 'public'
assert TABLES.exists() and PUBLIC_FIGURES.exists()
with (TABLES / '02_core_financial_summary.csv').open(encoding='utf-8') as f:
    core_summary = list(csv.DictReader(f))
# Public walkthrough intentionally leaves outputs unstored.

## 3. ตัวแปรสำคัญ

ตัวแปรหลัก ได้แก่ `TotalIncome`, `TotalExpense`, `NetBalance`, `ExpenseRatio`, `SavingAmount` และ `MoneySufficiency_Code` ดูรายละเอียดได้ที่ [พจนานุกรมข้อมูล](../docs/data_dictionary.md)

## 4. การทำความสะอาดข้อมูล

รูปแบบจำนวนเงินที่ชัดเจนถูกทำให้เป็นมาตรฐาน ค่าสูญหายทางการเงินจะไม่ถูกแทนด้วยศูนย์โดยปริยาย ดู [บันทึกการทำความสะอาดข้อมูล](../docs/cleaning_log.md)

## 5. ตัวชี้วัดทางการเงิน

`NetBalance = TotalIncome − TotalExpense` และ `ExpenseRatio = TotalExpense / TotalIncome × 100` รหัส `MoneySufficiency_Code` ที่สูงขึ้นหมายถึงความไม่เพียงพอทางการเงินมากขึ้น

## 6. การวิเคราะห์เชิงสำรวจ (Exploratory Data Analysis: EDA)

![กลุ่มความเพียงพอทางการเงิน](../outputs/figures/public/01_money_sufficiency_public.png)

![สรุปตัวแปรการเงิน](../outputs/figures/public/02_financial_summary_public.png)

![หมวดรายจ่าย](../outputs/figures/public/03_expense_categories_public.png)

![สรุป NetBalance](../outputs/figures/public/04_net_balance_summary_public.png)

ภาพทั้งหมดแสดงเฉพาะผลสรุปแบบ aggregate และไม่มีจุดสังเกตการณ์รายบุคคล

## 7. ค่าที่มีอิทธิพลและการวิเคราะห์ความไว

มีค่า `ExpenseRatio` ที่ผ่านการตรวจสอบและยังคงอยู่ในการวิเคราะห์หลัก การวิเคราะห์ความไวตัดค่านี้ออกชั่วคราวเพื่อประเมินอิทธิพลเท่านั้น ไม่ใช่ข้อมูลที่แก้ไขแล้วหรือผลที่ควรเลือกใช้แทน

![สรุป sensitivity ของ ExpenseRatio](../outputs/figures/public/06_expense_ratio_sensitivity_public.png)

In [ ]:
with (TABLES / '12_expense_ratio_sensitivity.csv').open(encoding='utf-8') as f:
    sensitivity_summary = list(csv.DictReader(f))
# Public walkthrough intentionally leaves outputs unstored.

## 8. สหสัมพันธ์แบบสเปียร์แมน (Spearman Correlation)

เลือกใช้ Spearman เพราะตัวแปรการเงินอาจเบ้และผลลัพธ์บางตัวเป็นข้อมูลอันดับ `SavingAmount × MoneySufficiency_Code`: rho=-0.370, n=30, permutation p=0.047, **ระดับอ่อน (Weak)** รหัส `MoneySufficiency_Code` ที่สูงขึ้นหมายถึงความไม่เพียงพอมากขึ้น ความสัมพันธ์ไม่ใช่เหตุและผล และ `TotalIncome × NetBalance` มี mathematical coupling เพราะ NetBalance มี TotalIncome เป็นส่วนประกอบ

![สรุปสหสัมพันธ์](../outputs/figures/public/07_correlation_summary_public.png)

## 9. ข้อจำกัดของรหัสพฤติกรรม

ไม่มี anchors ของ Likert ดั้งเดิม รหัสพฤติกรรมจึงเป็นเพียงรหัสตัวเลข และไม่ตีความว่ารหัสสูง/ต่ำหมายถึงพฤติกรรมดีขึ้น แย่ลง เห็นด้วยมากขึ้น หรือเกิดบ่อยขึ้น

In [ ]:
with (TABLES / '14_money_sufficiency_correlations.csv').open(encoding='utf-8') as f:
    money_sufficiency_correlations = list(csv.DictReader(f))
# Public walkthrough intentionally leaves outputs unstored.

## 10. ผลการวิเคราะห์สำคัญ

อาหารและที่พักเป็นหมวดรายจ่ายสำคัญของกรณีที่มีข้อมูลรายจ่ายครบ `NetBalance` ใน 26 กรณีที่คำนวณได้ มีบวก 18 ศูนย์ 2 และลบ 6 เงินออมมีความสัมพันธ์เชิงสำรวจกับรหัสความไม่เพียงพอที่ต่ำลง

## 11. ข้อจำกัด

กลุ่มตัวอย่างสมัครใจ มีขนาดเล็ก เป็นข้อมูล self-reported และจำนวนข้อมูลที่ใช้ได้แตกต่างกันตามตัวแปร สหสัมพันธ์เป็นการสำรวจและไม่ใช่เหตุผลเชิงสาเหตุ การทำซ้ำผลระดับผู้ตอบไม่พร้อมใช้งานสาธารณะ

## 12. สรุป

ชุดสาธารณะนี้เป็น walkthrough การวิเคราะห์แบบ aggregate ที่ปลอดภัยต่อความเป็นส่วนตัว พร้อมตารางผลสรุปและเอกสารประกอบ โดยไม่เปิดเผยระเบียนระดับผู้ตอบ